In [10]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def compute_loss_influence_summary(
    df,
    loss_col="Training_Loss",
    influence_col="Influence",
    label_col="label",
    top_fraction=0.10
):
    """
    Compute the statistical summary requested for the loss–influence analysis.

    Returns:
        dict containing:
        - Spearman(loss, raw influence)
        - Spearman(loss, absolute influence)
        - Proportion of high-loss samples in positive top-10%
        - Proportion of high-loss samples in negative top-10%
        - Proportion of high-loss samples in absolute top-10%
    """

    required = {loss_col, influence_col, label_col}
    missing = required.difference(df.columns)

    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    work = (
        df[[loss_col, influence_col, label_col]]
        .dropna()
        .copy()
    )
    n = len(work)

    if n == 0:
        raise ValueError("No valid samples remain after removing missing values.")

    k = max(1, int(np.ceil(n * top_fraction)))

    # 1. Spearman correlation: loss vs raw signed influence
    rho_raw, p_raw = spearmanr(
        work[loss_col],
        work[influence_col]
    )

    work_neg = work[work[influence_col] < 0].copy()
    
    work_neg["Abs_Influence"] = work_neg[influence_col].abs()
    
    rho_neg_abs, p_neg_abs = spearmanr(
        work_neg[loss_col],
        work_neg["Abs_Influence"]
    )
    
    # -------------------------
    # Positive influence (use raw score)
    # -------------------------
    work_pos = work[work[influence_col] > 0]
    
    rho_pos, p_pos = spearmanr(
        work_pos[loss_col],
        work_pos[influence_col]
    )
        

    work_y0 = work[work[label_col] == 0]

    rho_y0, p_y0 = spearmanr(
        work_y0[loss_col],
        work_y0[influence_col]
    )
    
    # Spearman correlation within label y = 1
    work_y1 = work[work[label_col] == 1]
    
    rho_y1, p_y1 = spearmanr(
        work_y1[loss_col],
        work_y1[influence_col]
    )

    # 2. Spearman correlation: loss vs influence magnitude
    work["Abs_Influence"] = work[influence_col].abs()

    rho_abs, p_abs = spearmanr(
        work[loss_col],
        work["Abs_Influence"]
    )

    # High-loss samples: top 10% by loss
    high_loss_ids = set(
        work.nlargest(k, loss_col).index
    )

    # Top 10% largest positive/raw influence scores
    positive_top_ids = set(
        work.nlargest(k, influence_col).index
    )

    # Top 10% most negative influence scores
    negative_top_ids = set(
        work.nsmallest(k, influence_col).index
    )

    # Top 10% largest influence magnitudes
    absolute_top_ids = set(
        work.nlargest(k, "Abs_Influence").index
    )

    # Denominator is the number of high-loss samples
    high_loss_count = len(high_loss_ids)

    prop_positive = (
        len(high_loss_ids & positive_top_ids)
        / high_loss_count
    )

    prop_negative = (
        len(high_loss_ids & negative_top_ids)
        / high_loss_count
    )

    prop_absolute = (
        len(high_loss_ids & absolute_top_ids)
        / high_loss_count
    )

    return {
        "Spearman(loss, I)": rho_raw,
        "Spearman p-value (raw)": p_raw,
    
        "Spearman(loss, I | y=0)": rho_y0,
        "Spearman p-value (y=0)": p_y0,
    
        "Spearman(loss, I | y=1)": rho_y1,
        "Spearman p-value (y=1)": p_y1,
    
        "Spearman(loss, |I|)": rho_abs,
        "Spearman p-value (abs)": p_abs,

        "Spearman(loss, |I| | I<0)": rho_neg_abs,
        "Spearman p-value (|I| | I<0)": p_neg_abs,
        
        "Spearman(loss, I | I>0)": rho_pos,
        "Spearman p-value (I | I>0)": p_pos,
    
        "High-loss in Pos. Top-10%": prop_positive,
        "High-loss in Neg. Top-10%": prop_negative,
        "High-loss in Abs. Top-10%": prop_absolute,
    
        "N": n,
        "K": k
    }

In [11]:
# Read files
adult_loss_df = pd.read_csv("Train_Loss_Train_Set_1_Adult.csv")
adult_if_df   = pd.read_csv("IF_Train_Set_1_Adult.csv")
adult_tc_df   = pd.read_csv("TC_Train_Set_1_Adult.csv")
adult_labels = pd.read_csv("train_labels_adult.csv")

# Merge
adult_if_data = adult_loss_df.merge(
    adult_if_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(adult_labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

adult_tc_data = adult_loss_df.merge(
    adult_tc_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(adult_labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

In [12]:
# Read files
loss_df = pd.read_csv("Train_Loss_Train_Set_1.csv")
if_df   = pd.read_csv("IF_Train_Set_1.csv")
tc_df   = pd.read_csv("TC_Train_Set_1.csv")
labels = pd.read_csv("train_labels.csv")

# Merge
if_data = loss_df.merge(
    if_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

tc_data = loss_df.merge(
    tc_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

In [13]:
# # Read files
# dia_loss_df = pd.read_csv("Diamonds_Train_Loss_Train_Set_1.csv")
# dia_if_df   = pd.read_csv("Diamonds_IF_Train_Set_1.csv")
# dia_tc_df   = pd.read_csv("Diamonds_TC_Train_Set_1.csv")
# dia_labels = pd.read_csv("train_labels_diamonds.csv")

# # Merge
# dia_if_data = dia_loss_df.merge(
#     dia_if_df[["Train_ID", "Score"]],
#     on="Train_ID"
# ).merge(dia_labels, on="Train_ID", how="left").rename(columns={
#     "Loss": "Training_Loss",
#     "Score": "Influence"
# })

# dia_tc_data = dia_loss_df.merge(
#     dia_tc_df[["Train_ID", "Score"]],
#     on="Train_ID"
# ).merge(dia_labels, on="Train_ID", how="left").rename(columns={
#     "Loss": "Training_Loss",
#     "Score": "Influence"
# })

In [14]:
datasets = {
    "SynA FOIF": if_data,
    "SynA TC": tc_data,
    "Adult_FOIF":adult_if_data,
    "Adult_TracIn":adult_tc_data,
    # "Dia_FOIF":dia_if_data,
    # "Dia_TracIn":dia_tc_data,

}

rows = []

for method, df in datasets.items():

    result = compute_loss_influence_summary(df)

    rows.append({
        "Method": method,
        "ρ(loss, I)": result["Spearman(loss, I)"],
        "ρ(loss, I | y=0)": result["Spearman(loss, I | y=0)"],
        "ρ(loss, I | y=1)": result["Spearman(loss, I | y=1)"],
        "ρ(loss, |I|)": result["Spearman(loss, |I|)"],
        "ρ(loss, |I| | I<0)": result["Spearman(loss, |I| | I<0)"],
        "ρ(loss, I | I>0)": result["Spearman(loss, I | I>0)"],
        "High-loss in Pos. Top10%": result["High-loss in Pos. Top-10%"],
        "High-loss in Neg. Top10%": result["High-loss in Neg. Top-10%"],
        "High-loss in Abs. Top10%": result["High-loss in Abs. Top-10%"],
    })

summary = pd.DataFrame(rows)

print(summary)

         Method  ρ(loss, I)  ρ(loss, I | y=0)  ρ(loss, I | y=1)  ρ(loss, |I|)  \
0     SynA FOIF    0.436403          0.462034          0.411593      0.866670   
1       SynA TC   -0.008686          0.700187         -0.691742      0.668394   
2    Adult_FOIF   -0.331823          0.174753         -0.504346      0.460189   
3  Adult_TracIn   -0.095337          0.738941         -0.828358      0.876041   

   ρ(loss, |I| | I<0)  ρ(loss, I | I>0)  High-loss in Pos. Top10%  \
0            0.932740          0.888169                  0.015000   
1            0.691742          0.700187                  0.305000   
2            0.456571          0.242198                  0.000000   
3            0.828358          0.738941                  0.017214   

   High-loss in Neg. Top10%  High-loss in Abs. Top10%  
0                  0.846000                  0.592000  
1                  0.445000                  0.531000  
2                  0.484462                  0.460765  
3                  0.724

In [15]:
check = if_data.copy()

check["Loss_Decile"] = pd.qcut(
    check["Training_Loss"],
    q=10,
    labels=False,
    duplicates="drop"
)

decile_summary = (
    check.groupby("Loss_Decile")
    .apply(
        lambda x: pd.Series({
            "Count": len(x),
            "Mean_Loss": x["Training_Loss"].mean(),
            "Median_Loss": x["Training_Loss"].median(),
            "Mean_Influence": x["Influence"].mean(),
            "Median_Influence": x["Influence"].median(),
            "Positive_Fraction": (x["Influence"] > 0).mean(),
            "Spearman_rho": spearmanr(
                x["Training_Loss"],
                x["Influence"]
            ).statistic,
            "Spearman_p": spearmanr(
                x["Training_Loss"],
                x["Influence"]
            ).pvalue
        })
    )
    .reset_index()
)

print(decile_summary)

   Loss_Decile   Count  Mean_Loss  Median_Loss  Mean_Influence  \
0            0  1000.0   0.001012     0.000961        0.001124   
1            1  1000.0   0.003479     0.003456        0.003128   
2            2  1000.0   0.007301     0.007210        0.005542   
3            3  1000.0   0.013394     0.013288        0.008521   
4            4  1000.0   0.023150     0.022917        0.012344   
5            5  1000.0   0.039829     0.038998        0.017452   
6            6  1000.0   0.071284     0.070566        0.024053   
7            7  1000.0   0.139540     0.133098        0.030579   
8            8  1000.0   0.341566     0.323494        0.027195   
9            9  1000.0   1.524341     1.150251       -0.132396   

   Median_Influence  Positive_Fraction  Spearman_rho     Spearman_p  
0          0.001110              1.000      0.952023   0.000000e+00  
1          0.003048              1.000      0.776985  8.364294e-203  
2          0.005429              1.000      0.679347  2.667898e

D:\Temp\ipykernel_21440\927430422.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [16]:
check = adult_if_data.copy()

check["Loss_Decile"] = pd.qcut(
    check["Training_Loss"],
    q=10,
    labels=False,
    duplicates="drop"
)

decile_summary = (
    check.groupby("Loss_Decile")
    .apply(
        lambda x: pd.Series({
            "Count": len(x),
            "Mean_Loss": x["Training_Loss"].mean(),
            "Median_Loss": x["Training_Loss"].median(),
            "Mean_Influence": x["Influence"].mean(),
            "Median_Influence": x["Influence"].median(),
            "Positive_Fraction": (x["Influence"] > 0).mean(),
            "Spearman_rho": spearmanr(
                x["Training_Loss"],
                x["Influence"]
            ).statistic,
            "Spearman_p": spearmanr(
                x["Training_Loss"],
                x["Influence"]
            ).pvalue
        })
    )
    .reset_index()
)

print(decile_summary)

   Loss_Decile   Count  Mean_Loss  Median_Loss  Mean_Influence  \
0            0  4473.0   0.151955     0.205809        0.020680   
1            1  4472.0   0.235931     0.238077        0.064526   
2            2  4472.0   0.247388     0.247392        0.146886   
3            3  4472.0   0.250810     0.250865        0.120953   
4            4  4472.0   0.253624     0.253544        0.108420   
5            5  4472.0   0.258014     0.257386        0.093810   
6            6  4472.0   0.272382     0.271601        0.107132   
7            7  4472.0   0.404298     0.342237        0.052191   
8            8  4472.0   1.313176     1.437259       -0.230780   
9            9  4473.0   1.563272     1.518418       -0.360498   

   Median_Influence  Positive_Fraction  Spearman_rho     Spearman_p  
0          0.016467           0.929577     -0.140351   4.067188e-21  
1          0.054401           0.836315      0.960304   0.000000e+00  
2          0.121672           0.999106     -0.271846   1.322007

D:\Temp\ipykernel_21440\2026999771.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [17]:
# check = dia_if_data.copy()

# check["Loss_Decile"] = pd.qcut(
#     check["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# )

# decile_summary = (
#     check.groupby("Loss_Decile")
#     .apply(
#         lambda x: pd.Series({
#             "Count": len(x),
#             "Mean_Loss": x["Training_Loss"].mean(),
#             "Median_Loss": x["Training_Loss"].median(),
#             "Mean_Influence": x["Influence"].mean(),
#             "Median_Influence": x["Influence"].median(),
#             "Positive_Fraction": (x["Influence"] > 0).mean(),
#             "Spearman_rho": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).statistic,
#             "Spearman_p": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).pvalue
#         })
#     )
#     .reset_index()
# )

# print(decile_summary)

In [18]:
# check = tc_data.copy()

# check["Loss_Decile"] = pd.qcut(
#     check["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# )

# decile_summary = (
#     check.groupby("Loss_Decile")
#     .apply(
#         lambda x: pd.Series({
#             "Count": len(x),
#             "Mean_Loss": x["Training_Loss"].mean(),
#             "Median_Loss": x["Training_Loss"].median(),
#             "Mean_Influence": x["Influence"].mean(),
#             "Median_Influence": x["Influence"].median(),
#             "Positive_Fraction": (x["Influence"] > 0).mean(),
#             "Spearman_rho": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).statistic,
#             "Spearman_p": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).pvalue
#         })
#     )
#     .reset_index()
# )

# print(decile_summary)